In [2]:
# ============================================================
# CELL 1 — SETUP AND LOAD THE BINARY FEATURE MATRIX
# Notebook: 05_model_training.ipynb (binary: Normal/Stress)
#
# The real IoT ring has no ground truth for a "pre-stress" state.
# WESAD's TSST condition maps directly to Stress in notebook 03's
# create_pss_labels() — no synthetic Pre-stress class is ever
# created, so there's nothing to relabel here. This notebook only
# needs to validate (LOSO) and train the deployed model.
#
# Does NOT touch or overwrite the 3-class artifacts
# (best_model_obj1.pkl / scaler_obj1.pkl) if those still exist from
# an earlier iteration of this project — this notebook only
# produces the binary artifacts (best_model_obj1_v2.pkl /
# scaler_obj1_v2.pkl) predictor.py loads for /predict.
# ============================================================

import numpy as np
import pandas as pd
import os
import sys
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
import joblib

COMPONENT_PATH = r"C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals"
sys.path.insert(0, os.path.join(COMPONENT_PATH, "src"))
from utils.config import RESULTS_PATH, MODELS_PATH

csv_path = os.path.join(RESULTS_PATH, "objective1", "wesad_features_selected.csv")
df = pd.read_csv(csv_path)
feature_cols = [c for c in df.columns if c not in ['label', 'subject_id']]

X        = df[feature_cols].values
y        = df['label'].values
subjects = df['subject_id'].values

print("=" * 65)
print("BINARY MODEL TRAINING (Normal vs Stress) — SETUP")
print("=" * 65)
print(f"Source: {csv_path}")
print(f"Shape:  {df.shape}")
print(f"Features: {len(feature_cols)}")
print()
print("LABEL DISTRIBUTION:")
for lid, name in {0: 'Normal', 1: 'Stress'}.items():
    count = int((y == lid).sum())
    print(f"  Class {lid} ({name:6s}): {count:4d} windows ({count/len(y)*100:.1f}%)")
print()
print(f"Subjects: {sorted(np.unique(subjects).tolist())}")
print()
print("Cell 1 complete — ready for LOSO validation")


Config loaded successfully
BINARY MODEL TRAINING (Normal vs Stress) — SETUP
Source: C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals\results\objective1\wesad_features_selected.csv
Shape:  (913, 25)
Features: 23

LABEL DISTRIBUTION:
  Class 0 (Normal):  587 windows (64.3%)
  Class 1 (Stress):  326 windows (35.7%)

Subjects: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]

Cell 1 complete — ready for LOSO validation


In [3]:
# ============================================================
# CELL 2 — LOSO CROSS-VALIDATION (leave-one-subject-out)
# Fit StandardScaler and XGBoost on 14 subjects, test on the 1
# held-out subject, repeat for all 15, pool predictions across
# every fold. class-imbalance handled via sample_weight='balanced'.
# ============================================================

XGB_PARAMS = dict(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, n_jobs=-1, verbosity=0
)

def run_loso_binary(X, y, subjects):
    unique_subjects = sorted(np.unique(subjects))
    all_true, all_pred = [], []

    for sid in unique_subjects:
        test_mask, train_mask = subjects == sid, subjects != sid
        X_train, y_train = X[train_mask], y[train_mask]
        X_test,  y_test  = X[test_mask],  y[test_mask]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)

        sample_weights = compute_sample_weight('balanced', y_train)
        model = xgb.XGBClassifier(**XGB_PARAMS)
        model.fit(X_train, y_train, sample_weight=sample_weights)

        y_pred = model.predict(X_test)
        all_true.extend(y_test.tolist())
        all_pred.extend(y_pred.tolist())

    all_true, all_pred = np.array(all_true), np.array(all_pred)
    return {
        'all_true': all_true, 'all_pred': all_pred,
        'accuracy': float(np.mean(all_true == all_pred)),
        'f1_macro': f1_score(all_true, all_pred, average='macro', zero_division=0),
        'f1_normal': f1_score(all_true, all_pred, labels=[0], average='macro', zero_division=0),
        'f1_stress': f1_score(all_true, all_pred, labels=[1], average='macro', zero_division=0),
    }


print("=" * 65)
print("LOSO CROSS-VALIDATION")
print("=" * 65)
result = run_loso_binary(X, y, subjects)

print(f"  Accuracy:   {result['accuracy']:.3f}")
print(f"  F1-macro:   {result['f1_macro']:.3f}")
print(f"  F1-Normal:  {result['f1_normal']:.3f}")
print(f"  F1-Stress:  {result['f1_stress']:.3f}")

print("\nDetailed report:")
print(classification_report(
    result['all_true'], result['all_pred'],
    target_names=['Normal', 'Stress'], digits=3
))

cm = confusion_matrix(result['all_true'], result['all_pred'])
print("Confusion matrix:")
print(f"                 Predicted")
print(f"                 Normal  Stress")
for i, row in enumerate(cm):
    lbl = ['Normal', 'Stress'][i]
    print(f"  Actual {lbl:7s}: {row[0]:6d}  {row[1]:6d}")

print("\nCell 2 complete")


LOSO CROSS-VALIDATION
  Accuracy:   0.851
  F1-macro:   0.842
  F1-Normal:  0.879
  F1-Stress:  0.805

Detailed report:
              precision    recall  f1-score   support

      Normal      0.917     0.845     0.879       587
      Stress      0.755     0.862     0.805       326

    accuracy                          0.851       913
   macro avg      0.836     0.853     0.842       913
weighted avg      0.859     0.851     0.853       913

Confusion matrix:
                 Predicted
                 Normal  Stress
  Actual Normal :    496      91
  Actual Stress :     45     281

Cell 2 complete


In [4]:
# ============================================================
# CELL 2.5 — ROC ANALYSIS & DECISION THRESHOLD SELECTION
#
# Cell 2's LOSO loop only ever called model.predict() (XGBoost's own
# implicit >0.5 cutoff on predict_proba) -- that cutoff was never actually
# validated against the data, it's just XGBoost's default. This cell pools
# each fold's held-out predict_proba() scores (same LOSO folds as Cell 2,
# same XGB_PARAMS, same StandardScaler-per-fold) into one out-of-fold ROC
# curve, and selects an operating threshold via Youden's J statistic
# (argmax of TPR - FPR) -- the standard principled choice absent a stated
# cost asymmetry between a missed Stress reading and a false Stress alarm.
#
# The result (STRESS_DECISION_THRESHOLD) is copied into
# backend/common/config.py and used by predictor.py's predict_window()/
# predict_raw() instead of the implicit prob_stress > prob_normal (0.5)
# comparison. Re-run this cell (and copy the printed threshold into
# config.py) whenever the model is retrained on new data.
# ============================================================

from sklearn.metrics import roc_curve, roc_auc_score

def run_loso_binary_proba(X, y, subjects):
    unique_subjects = sorted(np.unique(subjects))
    all_true, all_prob = [], []

    for sid in unique_subjects:
        test_mask, train_mask = subjects == sid, subjects != sid
        X_train, y_train = X[train_mask], y[train_mask]
        X_test,  y_test  = X[test_mask],  y[test_mask]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)

        sample_weights = compute_sample_weight('balanced', y_train)
        model = xgb.XGBClassifier(**XGB_PARAMS)
        model.fit(X_train, y_train, sample_weight=sample_weights)

        prob = model.predict_proba(X_test)[:, 1]  # P(Stress)
        all_true.extend(y_test.tolist())
        all_prob.extend(prob.tolist())

    return np.array(all_true), np.array(all_prob)


print("=" * 65)
print("ROC ANALYSIS (pooled out-of-fold LOSO probabilities)")
print("=" * 65)
roc_true, roc_prob = run_loso_binary_proba(X, y, subjects)

auc = roc_auc_score(roc_true, roc_prob)
fpr, tpr, thresholds = roc_curve(roc_true, roc_prob)
youden_j = tpr - fpr
best_idx = int(np.argmax(youden_j))
STRESS_DECISION_THRESHOLD = float(thresholds[best_idx])

print(f"n = {len(roc_true)} pooled out-of-fold predictions")
print(f"AUC = {auc:.4f}")
print(f"Youden's J-optimal threshold = {STRESS_DECISION_THRESHOLD:.4f}")
print(f"  at that point: TPR={tpr[best_idx]:.4f}  FPR={fpr[best_idx]:.4f}  J={youden_j[best_idx]:.4f}")

idx_half = int(np.argmin(np.abs(thresholds - 0.5)))
print(f"\nFor comparison, the old implicit 0.5 cutoff:")
print(f"  threshold={thresholds[idx_half]:.4f}  TPR={tpr[idx_half]:.4f}  FPR={fpr[idx_half]:.4f}  J={youden_j[idx_half]:.4f}")

print(f"\n>>> Copy STRESS_DECISION_THRESHOLD = {STRESS_DECISION_THRESHOLD:.4f} into backend/common/config.py <<<")
print("\nCell 2.5 complete")


ROC ANALYSIS (pooled out-of-fold LOSO probabilities)
n = 913 pooled out-of-fold predictions
AUC = 0.9382
Youden's J-optimal threshold = 0.7141
  at that point: TPR=0.8497  FPR=0.1210  J=0.7287

For comparison, the old implicit 0.5 cutoff:
  threshold=0.5025  TPR=0.8620  FPR=0.1550  J=0.7069

>>> Copy STRESS_DECISION_THRESHOLD = 0.7141 into backend/common/config.py <<<

Cell 2.5 complete


In [4]:
# ============================================================
# CELL 3 — FINAL TRAINING ON ALL SUBJECTS, SAVE FOR DEPLOYMENT
# Standard practice once LOSO has validated the approach — retrain
# on all 15 subjects combined for the deployed model (more data
# only helps). Saves the artifacts backend/physiological/
# predictor.py loads for /predict.
# ============================================================

scaler_final = StandardScaler()
X_scaled = scaler_final.fit_transform(X)
sample_weights = compute_sample_weight('balanced', y)

model_final = xgb.XGBClassifier(**XGB_PARAMS)
model_final.fit(X_scaled, y, sample_weight=sample_weights)

os.makedirs(MODELS_PATH, exist_ok=True)
model_path    = os.path.join(MODELS_PATH, "best_model_obj1_v2.pkl")
scaler_path   = os.path.join(MODELS_PATH, "scaler_obj1_v2.pkl")
features_path = os.path.join(MODELS_PATH, "feature_cols_obj1_v2.pkl")

joblib.dump(model_final, model_path)
joblib.dump(scaler_final, scaler_path)
joblib.dump(feature_cols, features_path)

print("=" * 65)
print("BINARY MODEL DEPLOYED (Normal vs Stress)")
print("=" * 65)
print(f"Trained on: {len(y)} windows (Normal={int((y==0).sum())}, Stress={int((y==1).sum())})")
print(f"LOSO accuracy: {result['accuracy']:.3f}   LOSO F1-macro: {result['f1_macro']:.3f}")
print()
print("Saved:")
print(f"  {model_path}")
print(f"  {scaler_path}")
print(f"  {features_path}")
print()
print("trending_toward_stress is unrelated to this notebook — it stays a")
print("runtime indicator computed by StressStateMachine in predictor.py")
print("(feature-direction voting over recent windows), never a trained class.")
print()
print("Next: backend/physiological/predictor.py loads these files at Flask")
print("startup and serves them via /predict — no code changes needed to")
print("pick up a rerun of this notebook, same file paths every time.")


BINARY MODEL DEPLOYED (Normal vs Stress)
Trained on: 913 windows (Normal=587, Stress=326)
LOSO accuracy: 0.851   LOSO F1-macro: 0.842

Saved:
  C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals\saved_models\best_model_obj1_v2.pkl
  C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals\saved_models\scaler_obj1_v2.pkl
  C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals\saved_models\feature_cols_obj1_v2.pkl

trending_toward_stress is unrelated to this notebook — it stays a
runtime indicator computed by StressStateMachine in predictor.py
(feature-direction voting over recent windows), never a trained class.

Next: backend/physiological/predictor.py loads these files at Flask
startup and serves them via /predict — no code changes needed to
pick up a rerun of this notebook, same file paths every time.
